# Full Solution: Customer Churn Prediction

This notebook solves the Telco Customer Churn classification assignment using **Logistic Regression, Decision Tree, and Random Forest**.

**Dataset:** [Telco Customer Churn – Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

Before running the notebook, download `WA_Fn-UseC_-Telco-Customer-Churn.csv` from Kaggle and place it in the same folder as this notebook.

The notebook uses a train/test split and fits all preprocessing steps on the training data only to avoid data leakage.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    classification_report
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [ ]:
# Keep the CSV file in the same directory as this notebook.
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

## 3. Basic dataset exploration

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn information:")
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
print("Missing values before cleaning:")
print(df.isna().sum().sort_values(ascending=False).head(10))

print("\\nDuplicate rows:", df.duplicated().sum())

print("\\nTarget distribution:")
print(df["Churn"].value_counts())
print("\\nTarget proportions:")
print(df["Churn"].value_counts(normalize=True).round(3))

### Visualizations

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.show()

In [ ]:
churn_by_contract = (
    df.groupby("Contract", observed=False)["Churn"]
      .apply(lambda x: (x == "Yes").mean())
      .sort_values(ascending=False)
)

plt.figure(figsize=(7, 4))
sns.barplot(x=churn_by_contract.index, y=churn_by_contract.values)
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract")
plt.ylabel("Churn rate")
plt.ylim(0, 1)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="MonthlyCharges", hue="Churn", bins=30,
             element="step", stat="density", common_norm=False)
plt.title("Monthly Charges Distribution by Churn")
plt.xlabel("Monthly charges")
plt.show()

### Brief EDA observations

Use the plots and printed summaries to write down what you notice. For example:
- Is the target balanced or imbalanced?
- Which contract type appears to have the highest churn rate?
- Do monthly charges look different for customers who churn and those who stay?

Do not assume these patterns prove causation.

## 4. Data cleaning

In [ ]:
# Work on a copy so the original DataFrame remains unchanged.
data = df.copy()

# customerID is an identifier, not a useful customer-behaviour feature.
data = data.drop(columns=["customerID"])

# TotalCharges may contain blank strings. Convert them to numeric;
# invalid/blank values become NaN and will be imputed later.
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")

# Convert target labels into 0 and 1.
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

print("Missing values after converting TotalCharges:")
print(data.isna().sum().sort_values(ascending=False).head())

print("\\nTarget values:")
print(data["Churn"].value_counts())

## 5. Separate features and target

In [ ]:
X = data.drop(columns="Churn")
y = data["Churn"]

# Identify column types before preprocessing.
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numeric_features)
print("\\nCategorical features:", categorical_features)

## 6. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

print("\\nTraining target proportions:")
print(y_train.value_counts(normalize=True).round(3))

print("\\nTesting target proportions:")
print(y_test.value_counts(normalize=True).round(3))

## 7. Preprocessing

Numerical columns:
- Fill missing values with the median.
- Standardize the values.

Categorical columns:
- Fill missing values with the most frequent category.
- Apply One-Hot Encoding.

**Important:** `fit_transform()` is used only on the training set. The test set receives `transform()` using the transformations learned from the training set.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

## 8. Logistic Regression

Logistic Regression is our baseline model. We use the scaled numerical features and one-hot encoded categorical features produced above.

In [ ]:
model_lr = LogisticRegression(
    C=1,
    max_iter=1000,
    random_state=42
)

model_lr.fit(X_train_processed, y_train)

y_pred_lr = model_lr.predict(X_test_processed)
y_prob_lr = model_lr.predict_proba(X_test_processed)[:, 1]

### Small experiment: regularization strength

In [ ]:
lr_experiment_results = []

for c_value in [0.1, 1, 10]:
    model = LogisticRegression(C=c_value, max_iter=1000, random_state=42)
    model.fit(X_train_processed, y_train)
    predictions = model.predict(X_test_processed)

    lr_experiment_results.append({
        "C": c_value,
        "Accuracy": accuracy_score(y_test, predictions),
        "F1-score": f1_score(y_test, predictions)
    })

pd.DataFrame(lr_experiment_results)

## 9. Decision Tree

In [ ]:
model_dt = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_dt.fit(X_train_processed, y_train)

y_pred_dt = model_dt.predict(X_test_processed)
y_prob_dt = model_dt.predict_proba(X_test_processed)[:, 1]

### Small experiment: maximum depth

In [ ]:
dt_experiment_results = []

for depth in [3, 5, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train_processed, y_train)
    predictions = model.predict(X_test_processed)

    dt_experiment_results.append({
        "max_depth": str(depth),
        "Accuracy": accuracy_score(y_test, predictions),
        "F1-score": f1_score(y_test, predictions)
    })

pd.DataFrame(dt_experiment_results)

## 10. Random Forest

In [ ]:
model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train_processed, y_train)

y_pred_rf = model_rf.predict(X_test_processed)
y_prob_rf = model_rf.predict_proba(X_test_processed)[:, 1]

### Small experiment: number of trees

In [ ]:
rf_experiment_results = []

for n_trees in [50, 100, 200]:
    model = RandomForestClassifier(
        n_estimators=n_trees,
        max_depth=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_processed, y_train)
    predictions = model.predict(X_test_processed)

    rf_experiment_results.append({
        "n_estimators": n_trees,
        "Accuracy": accuracy_score(y_test, predictions),
        "F1-score": f1_score(y_test, predictions)
    })

pd.DataFrame(rf_experiment_results)

## 11. Model evaluation

In [ ]:
models = {
    "Logistic Regression": (model_lr, y_pred_lr, y_prob_lr),
    "Decision Tree": (model_dt, y_pred_dt, y_prob_dt),
    "Random Forest": (model_rf, y_pred_rf, y_prob_rf)
}

results = []

for model_name, (model, predictions, probabilities) in models.items():
    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1-score": f1_score(y_test, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })

comparison_df = pd.DataFrame(results).set_index("Model")
comparison_df.round(3)

### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (model_name, (_, predictions, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, predictions)
    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Stayed (0)", "Churned (1)"]
    )
    display.plot(ax=ax, colorbar=False)
    ax.set_title(model_name)

plt.tight_layout()
plt.show()

### Classification reports

In [ ]:
for model_name, (_, predictions, _) in models.items():
    print("=" * 60)
    print(model_name)
    print(classification_report(
        y_test,
        predictions,
        target_names=["Stayed", "Churned"],
        zero_division=0
    ))

## 12. Conclusions

Answer these questions using your actual results:

1. Which model performed best? Which evaluation metric supports your conclusion?
2. Why can a model have high accuracy but relatively low recall?
3. What happens when you increase the maximum depth of a Decision Tree?
4. Why might Random Forest perform better than a single Decision Tree?

### Points to remember

- Accuracy alone can be misleading when the target classes are imbalanced.
- Recall is important when the goal is to identify as many customers who may churn as possible.
- A deeper Decision Tree can learn more complex patterns, but it may also overfit.
- Random Forest combines many trees and often reduces the instability of a single tree.
- Logistic Regression benefits from feature scaling. Tree-based models generally do not require it, although the same preprocessed data was used here to keep the exercise simple.

**Note:** The experiment tables use the same held-out test set for simplicity in this introductory assignment. For a more rigorous model-selection workflow, hyperparameters should be selected using cross-validation on the training set, and the test set should be used only for final evaluation.